# Cue-based behavioral models

Four analyses, matching the "Cue-based regression models" / "ARX(1) baseline" paragraphs
of the paper:

1. **Group-level ridge regression** — pooled across participants, one shared coefficient set.
2. **Subject-level multinomial logistic regression** — fit separately per participant.
3. **Static per-cue baseline** — non-parametric per-subject per-cue mean; the empirical
   benchmark for the noise-ceiling analysis.
4. **ARX(1) baseline** — per-subject ridge regression with cues + first-order choice history.

All four use the same 9-dimensional trial-wise cue vector $\boldsymbol\phi_t$ (4 identity
one-hot + 5 expression-level one-hot indicators), trained on each participant's first 60
trials and evaluated on the held-out last 20. Models 2 and 4 additionally report
identity-only (4-dim) and expression-only (5-dim) cue subsets alongside the full 9-dim
version, since that breakdown was useful during development; the full 9-dim version is the
one reported in the paper.

*Exploratory material that isn't one of these four (an RNN-comparison scratch section and
an AR(5)/PCA social-vs-non-social exploration) has been moved to
`Logistic regression and AR - exploratory.ipynb`.*

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

N_SUBJECTS = 32
N_TRAIN = 60   # trials 1-60
N_TEST = 20    # trials 61-80 (held out)

# Cue vector layout: 4 identity one-hot + 5 expression-level one-hot = 9 dims total
CUE_SUBSETS = {
    "identity (4)": slice(0, 4),
    "expression (5)": slice(4, 9),
    "full (9)": slice(0, 9),
}

REPO_ROOT = Path.cwd().resolve().parents[1]  # comparisons/baseline_models -> repo root
DATA_DIR = REPO_ROOT / "data"

# invests: (32, 80) trial-wise investment (1-5, NaN for missing);
# inputs:  (32, 80, 9) trial-wise cue vector (the raw arrays carry 160 rows -
# alternating decision/non-decision rows - and 11 input columns; ::2 and :9 select
# the 80 real decision trials and the 9-dim cue vector, see "Stack Data.ipynb")
invests_full = np.load(DATA_DIR / "stacked_invests.npy")[:, ::2, 0]      # (32, 80)
inputs_full = np.load(DATA_DIR / "stacked_inputs.npy")[:, ::2, :9]       # (32, 80, 9)

assert invests_full.shape == (N_SUBJECTS, N_TRAIN + N_TEST)
assert inputs_full.shape == (N_SUBJECTS, N_TRAIN + N_TEST, 9)

## 1. Group-level ridge regression

Ridge-regularized ($L_2$) linear regression pooled across participants: a single shared
coefficient set, no subject-specific parameters. Fit on trials 1-60 pooled over all
subjects, evaluated on held-out trials 61-80 (again pooled, but we also keep per-subject
predictions for the model-comparison section below).

In [2]:
X_train = inputs_full[:, :N_TRAIN, :].reshape(-1, 9)
y_train = invests_full[:, :N_TRAIN].reshape(-1)
train_mask = ~np.isnan(y_train)
X_train, y_train = X_train[train_mask], y_train[train_mask]

group_model = Ridge(alpha=1.0)
group_model.fit(X_train, y_train)

# Per-subject held-out predictions (same shared coefficients, evaluated per subject
# so we get per-subject MAE/corr for the comparison section)
maes_group, cors_group = [], []
for subj in range(N_SUBJECTS):
    X_test = inputs_full[subj, N_TRAIN:, :]
    y_test = invests_full[subj, N_TRAIN:]
    test_mask = ~np.isnan(y_test)
    X_test, y_test = X_test[test_mask], y_test[test_mask]

    y_pred = np.clip(np.round(group_model.predict(X_test)), 1, 5)
    maes_group.append(mean_absolute_error(y_test, y_pred))
    cors_group.append(pearsonr(y_test, y_pred)[0] if np.std(y_pred) > 0 else np.nan)

maes_group, cors_group = np.array(maes_group), np.array(cors_group)
print(f"Group-level ridge  — Avg MAE: {maes_group.mean():.3f}, Avg Corr: {np.nanmean(cors_group):.3f}")

Group-level ridge  — Avg MAE: 1.027, Avg Corr: 0.602


## 2. Subject-level multinomial logistic regression

$$p(o_t = k \mid \boldsymbol\phi_t) = \frac{\exp(\boldsymbol\beta_k^\top \boldsymbol\phi_t + \beta_{0,k})}
{\sum_{k'=1}^{5} \exp(\boldsymbol\beta_{k'}^\top \boldsymbol\phi_t + \beta_{0,k'})}$$

Fit separately per participant on trials 1-60, evaluated on trials 61-80. Prediction is
the most probable investment level (argmax), which is already a discrete label. Reported
for all three cue subsets; **full (9) is the version reported in the paper.**

In [3]:
def fit_subject_level(model_fn, cue_slice, use_lag=False):
    """Fit `model_fn()` (fresh estimator) separately per subject on trials 1..N_TRAIN,
    evaluate on the held-out N_TEST trials. If use_lag, appends a_{t-1} as an extra
    regressor (for the ARX(1) baseline); model_fn must then be a regressor, not a
    classifier."""
    maes, cors = [], []
    for subj in range(N_SUBJECTS):
        y = invests_full[subj]
        X_cue = inputs_full[subj, :, cue_slice]

        if use_lag:
            X_rows, y_rows, t_idx = [], [], []
            for t in range(1, len(y)):
                if np.isnan(y[t]) or np.isnan(y[t - 1]):
                    continue
                X_rows.append(np.concatenate([X_cue[t], [y[t - 1]]]))
                y_rows.append(y[t])
                t_idx.append(t)
            X_all, y_all = np.array(X_rows), np.array(y_rows)
            # split by original trial index, not row count, so any dropped/NaN
            # trials don't shift the train/test boundary
            t_split = sum(1 for t in t_idx if t < N_TRAIN)
        else:
            train_mask_t = ~np.isnan(y)
            X_all, y_all = X_cue[train_mask_t], y[train_mask_t]
            # re-derive split index in the masked series
            t_split = int(train_mask_t[:N_TRAIN].sum())

        X_train, y_train = X_all[:t_split], y_all[:t_split]
        X_test, y_test = X_all[t_split:], y_all[t_split:]

        if len(y_train) == 0 or len(y_test) == 0:
            maes.append(np.nan); cors.append(np.nan)
            continue
        if not use_lag and len(np.unique(y_train)) < 2:
            maes.append(np.nan); cors.append(np.nan)
            continue

        model = model_fn()
        model.fit(X_train, y_train.astype(int) if not use_lag else y_train)
        y_pred = model.predict(X_test)
        if use_lag:
            y_pred = np.clip(np.round(y_pred), 1, 5)

        maes.append(mean_absolute_error(y_test, y_pred))
        cors.append(pearsonr(y_test, y_pred)[0] if (np.std(y_test) > 0 and np.std(y_pred) > 0) else np.nan)

    return np.array(maes), np.array(cors)


logreg_results = {}
for name, sl in CUE_SUBSETS.items():
    maes, cors = fit_subject_level(lambda: LogisticRegression(max_iter=1000), sl)
    logreg_results[name] = (maes, cors)
    print(f"Subject-level logistic regression [{name:>14}] — "
          f"Avg MAE: {np.nanmean(maes):.3f}, Avg Corr: {np.nanmean(cors):.3f}")

maes_logreg_subj, cors_logreg_subj = logreg_results["full (9)"]

Subject-level logistic regression [  identity (4)] — Avg MAE: 0.794, Avg Corr: 0.534


Subject-level logistic regression [expression (5)] — Avg MAE: 1.211, Avg Corr: 0.160


Subject-level logistic regression [      full (9)] — Avg MAE: 0.789, Avg Corr: 0.544


## 3. Static per-cue baseline (noise ceiling)

Non-parametric reference with no free parameters beyond the empirical per-cue means, and
no temporal structure. For each participant, average the observed investments over all
*training* trials sharing the same cue combination $\boldsymbol\phi$, and use that value
as the prediction for every *test* trial presenting the same combination:

$$\hat o_t = \frac{1}{|\mathcal T_{\boldsymbol\phi_t}|} \sum_{s \in \mathcal T_{\boldsymbol\phi_t}} o_s,
\qquad \mathcal T_{\boldsymbol\phi} = \{s \in \text{train} : \boldsymbol\phi_s = \boldsymbol\phi\}$$

falling back to the participant's overall training-period mean for cue combinations not
encountered during training. This is the empirical benchmark for the noise-ceiling
analysis: its held-out error quantifies how far behavior can be predicted from a fixed,
subject-specific cue-response mapping alone, with no model beyond that.

In [4]:
def cue_combo_id(phi9):
    """Collapse the 9-dim one-hot cue vector to a single combo id in [0, 20) via
    (identity index, expression index)."""
    identity_idx = phi9[:, :4].argmax(axis=-1)
    expression_idx = phi9[:, 4:9].argmax(axis=-1)
    return identity_idx * 5 + expression_idx


maes_static, cors_static = [], []
for subj in range(N_SUBJECTS):
    y = invests_full[subj]
    combo = cue_combo_id(inputs_full[subj])

    y_train, combo_train = y[:N_TRAIN], combo[:N_TRAIN]
    y_test, combo_test = y[N_TRAIN:], combo[N_TRAIN:]

    train_valid = ~np.isnan(y_train)
    overall_train_mean = y_train[train_valid].mean()
    per_combo_mean = {
        c: y_train[train_valid & (combo_train == c)].mean()
        for c in np.unique(combo_train[train_valid])
    }

    test_valid = ~np.isnan(y_test)
    y_true = y_test[test_valid]
    y_pred = np.array([per_combo_mean.get(c, overall_train_mean) for c in combo_test[test_valid]])

    maes_static.append(mean_absolute_error(y_true, y_pred))
    cors_static.append(pearsonr(y_true, y_pred)[0] if (np.std(y_true) > 0 and np.std(y_pred) > 0) else np.nan)

maes_static, cors_static = np.array(maes_static), np.array(cors_static)
print(f"Static per-cue baseline (noise ceiling) — Avg MAE: {maes_static.mean():.3f}, Avg Corr: {np.nanmean(cors_static):.3f}")

Static per-cue baseline (noise ceiling) — Avg MAE: 0.804, Avg Corr: 0.636


## 4. ARX(1) baseline

$$\hat a_t = b_0 + b_1 \cdot a_{t-1} + \boldsymbol\beta^\top \boldsymbol\phi_t$$

Per-participant ridge regression combining the current cue configuration with a
first-order dependence on the participant's previous choice. Reported for all three cue
subsets; **full (9) is the version reported in the paper.**

In [5]:
arx_results = {}
for name, sl in CUE_SUBSETS.items():
    maes, cors = fit_subject_level(lambda: Ridge(alpha=1.0), sl, use_lag=True)
    arx_results[name] = (maes, cors)
    print(f"ARX(1) [{name:>14}] — Avg MAE: {np.nanmean(maes):.3f}, Avg Corr: {np.nanmean(cors):.3f}")

maes_arx1, cors_arx1 = arx_results["full (9)"]

ARX(1) [  identity (4)] — Avg MAE: 0.813, Avg Corr: 0.548


ARX(1) [expression (5)] — Avg MAE: 1.097, Avg Corr: 0.162
ARX(1) [      full (9)] — Avg MAE: 0.764, Avg Corr: 0.600


## Model comparison & noise ceiling

Per-subject MAE/correlation for all four models, plus paired comparisons of each
regression-based model against the static per-cue baseline (the noise ceiling).

In [6]:
summary = pd.DataFrame({
    "Group-level ridge": {"MAE": maes_group.mean(), "Corr": np.nanmean(cors_group)},
    "Subject-level logreg": {"MAE": np.nanmean(maes_logreg_subj), "Corr": np.nanmean(cors_logreg_subj)},
    "Static per-cue (ceiling)": {"MAE": maes_static.mean(), "Corr": np.nanmean(cors_static)},
    "ARX(1)": {"MAE": np.nanmean(maes_arx1), "Corr": np.nanmean(cors_arx1)},
}).T
summary

,MAE,Corr
Group-level ridge,1.026553,0.601936
Subject-level logreg,0.788706,0.543946
Static per-cue (ceiling),0.804230,0.635857
ARX(1),0.763715,0.599842


In [7]:
def paired_model_comparison(a, b, label="metric", fisher_z=False, n_boot=10000, seed=0):
    """
    Paired comparison of per-subject metric between two models.
    a, b : array-like, shape (n_subjects,) (a - b convention: positive diff means a > b)
    fisher_z : set True for correlation coefficients (applies atanh transform
               before the parametric test; Wilcoxon still uses raw values)
    """
    from scipy import stats
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    assert a.shape == b.shape, "Must be paired (same subjects, same order)"
    n = len(a)

    diff_raw = a - b
    if fisher_z:
        # clip away from +-1 -- a perfect per-subject correlation (possible with only
        # N_TEST=20 held-out trials) would otherwise send arctanh to +-inf
        za, zb = np.arctanh(np.clip(a, -0.999999, 0.999999)), np.arctanh(np.clip(b, -0.999999, 0.999999))
        diff_test = za - zb
    else:
        diff_test = diff_raw

    t_stat, p_ttest = stats.ttest_rel(a if not fisher_z else za, b if not fisher_z else zb)
    dz = diff_test.mean() / diff_test.std(ddof=1)

    try:
        w_stat, p_wilcoxon = stats.wilcoxon(a, b, zero_method="wilcox")
    except ValueError:
        w_stat, p_wilcoxon = np.nan, np.nan

    n_eff = np.sum(diff_raw != 0)
    if n_eff > 0 and not np.isnan(w_stat):
        r_rb = 1 - (2 * w_stat) / (n_eff * (n_eff + 1) / 2)
    else:
        r_rb = np.nan

    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    boot_means = diff_raw[idx].mean(axis=1)
    ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])

    print(f"--- {label} (n={n}) ---")
    print(f"Mean diff (a-b): {diff_raw.mean():.4f}  [95% CI {ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"Paired t-test:   t={t_stat:.3f}, p={p_ttest:.4g}, d_z={dz:.3f}")
    print(f"Wilcoxon:        W={w_stat:.3f}, p={p_wilcoxon:.4g}, r_rb={r_rb:.3f}")

    return dict(mean_diff=diff_raw.mean(), ci=(ci_lo, ci_hi), t=t_stat, p_t=p_ttest, dz=dz,
                W=w_stat, p_wilcoxon=p_wilcoxon, r_rb=r_rb)


for name, (maes, cors) in [("Group-level ridge", (maes_group, cors_group)),
                            ("Subject-level logreg", (maes_logreg_subj, cors_logreg_subj)),
                            ("ARX(1)", (maes_arx1, cors_arx1))]:
    print(f"\n=== {name} vs. static per-cue baseline (ceiling) ===")
    paired_model_comparison(maes, maes_static, label=f"{name} vs ceiling — MAE")


=== Group-level ridge vs. static per-cue baseline (ceiling) ===
--- Group-level ridge vs ceiling — MAE (n=32) ---
Mean diff (a-b): 0.2223  [95% CI 0.1324, 0.3158]
Paired t-test:   t=4.612, p=6.518e-05, d_z=0.815
Wilcoxon:        W=63.000, p=6.03e-05, r_rb=0.761

=== Subject-level logreg vs. static per-cue baseline (ceiling) ===
--- Subject-level logreg vs ceiling — MAE (n=32) ---
Mean diff (a-b): -0.0155  [95% CI -0.1151, 0.0757]
Paired t-test:   t=-0.314, p=0.7556, d_z=-0.056
Wilcoxon:        W=258.000, p=0.9192, r_rb=0.023

=== ARX(1) vs. static per-cue baseline (ceiling) ===
--- ARX(1) vs ceiling — MAE (n=32) ---
Mean diff (a-b): -0.0405  [95% CI -0.0796, -0.0001]
Paired t-test:   t=-1.941, p=0.06141, d_z=-0.343
Wilcoxon:        W=150.000, p=0.03249, r_rb=0.432


## RNN model

In [8]:
test_corrs=[0.326644  , 0.79112216, 0.44541755, 0.83288415, 0.27599078,
       0.74952002, 0.82687419, 0.88982067, 0.69724762, 0.68865435,
       0.85525662, 0.51130512, 0.25492496, 0.645657  , 0.21453773,
       0.49742488, 0.88041745, 0.50395263, 0.87173454, 0.76944422,
       0.48749802, 0.9327087 , 0.79211803, 0.58456658, 0.89479249,
       0.17191038, 0.74186099, 0.95575879, 0.6107136 , 0.68865004,
       0.85393235, 0.66831888]

In [9]:
test_errors=[1.45      , 0.5       , 0.9       , 0.36842105, 1.36842105,
       0.6       , 0.8       , 0.63157895, 0.68421053, 0.5       ,
       0.52631579, 0.84210526, 0.9       , 0.45      , 0.7       ,
       1.        , 0.65      , 0.6       , 0.7       , 0.35      ,
       0.31578947, 0.35      , 0.42105263, 1.26315789, 0.55      ,
       0.95      , 0.7       , 0.7       , 0.6       , 0.4       ,
       0.55      , 0.95      ]

### Each main model vs. the RNN

Paired per-subject comparison (same `paired_model_comparison` used for the noise-ceiling
comparisons above) of MAE and Fisher-z'd correlation against the RNN's `test_errors` /
`test_corrs`.

In [10]:
test_errors = np.asarray(test_errors, dtype=float)
test_corrs = np.asarray(test_corrs, dtype=float)

for name, (maes, cors) in [
    ("Group-level ridge", (maes_group, cors_group)),
    ("Subject-level logreg", (maes_logreg_subj, cors_logreg_subj)),
    ("Static per-cue (ceiling)", (maes_static, cors_static)),
    ("ARX(1)", (maes_arx1, cors_arx1)),
]:
    print(f"\n=== {name} vs. RNN ===")
    paired_model_comparison(np.asarray(maes), test_errors, label=f"{name} vs RNN — MAE")
    paired_model_comparison(np.asarray(cors), test_corrs, label=f"{name} vs RNN — Corr", fisher_z=True)


=== Group-level ridge vs. RNN ===
--- Group-level ridge vs RNN — MAE (n=32) ---
Mean diff (a-b): 0.3306  [95% CI 0.2130, 0.4510]
Paired t-test:   t=5.360, p=7.645e-06, d_z=0.948
Wilcoxon:        W=34.500, p=4.635e-05, r_rb=0.852
--- Group-level ridge vs RNN — Corr (n=32) ---
Mean diff (a-b): -0.0516  [95% CI -0.1367, 0.0268]
Paired t-test:   t=-1.514, p=0.1401, d_z=-0.268
Wilcoxon:        W=207.000, p=0.2951, r_rb=0.216

=== Subject-level logreg vs. RNN ===
--- Subject-level logreg vs RNN — MAE (n=32) ---
Mean diff (a-b): 0.0927  [95% CI 0.0133, 0.1779]
Paired t-test:   t=2.167, p=0.03806, d_z=0.383
Wilcoxon:        W=126.000, p=0.0284, r_rb=0.458
--- Subject-level logreg vs RNN — Corr (n=32) ---
Mean diff (a-b): -0.1095  [95% CI -0.1765, -0.0461]
Paired t-test:   t=-0.125, p=0.9016, d_z=-0.022
Wilcoxon:        W=100.000, p=0.001548, r_rb=0.621

=== Static per-cue (ceiling) vs. RNN ===
--- Static per-cue (ceiling) vs RNN — MAE (n=32) ---
Mean diff (a-b): 0.1083  [95% CI 0.0389, 0.1887